In [30]:
#%%
import math
import numpy as np

import inspect
import importlib

import scipy.interpolate
from scipy import stats
from scipy.stats import norm

import matplotlib
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
from matplotlib import ticker, cm
from matplotlib import patches as mpatches
import matplotlib.lines as mlines
from matplotlib import colormaps

import seaborn as sns


plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.family'] = 'STIXGeneral'
plt.rcParams['font.size'] = 16

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.offline import plot
pio.renderers.default = "svg"

import sys
import os
parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)
#sys.path.append(os.path.dirname(parent_directory))
#sys.path.append(os.path.dirname(os.path.dirname(parent_directory)))

from IoT_node_models.Energy_model                        import *
from IoT_node_models.Energy_model.Wireless_communication import *
from IoT_node_models.Hardware_Modules                    import *
from IoT_node_models.Analysis                            import *

from fft_architecture_exploration                        import *

path_to_save_svg = "Saved_Data"


rng = np.random.default_rng(42)

In [31]:
dictColor = {"DeepBlue"     :"#095256",
             "LightBlue"    :"#01a7c2",
             "UnitedBlue"   :"#6290c8",
             "CerulBlue"    :"#6096ba",
             "Green"        :"#44af69",
             "LightRed"     :"#f8333c",
             "DeepRed"      :"#a22c29",
             "Orange"       :"#fcab10",
             "Sandy"        :"#fc9e4F",
             "Olivine"      :"#9ab87a",
             "CarolinaBlue" :"#009DDC"}
listColor = list(dictColor.values())

In [32]:
def F_p_kWh_to_p_J (F_p_kWh):

    return F_p_kWh / 3600000 

def F_p_mAh_to_p_kWh (F_p_mAh, V):

    return F_p_mAh * 1e6 / (V)

def F_p_mAh_to_p_J (F_p_mAh, V):

    return F_p_mAh / (3600/1000 * V)

In [40]:
print( 119.2*0.0007)
print(250/10000 * 14.4 )
print(250/10000 * 14.4  + 119.2*0.0007)

0.08344
0.36000000000000004
0.44344000000000006


In [33]:
F_EH    = np.array([0.125,0.235,0.35,0.47])
P_mw_EH = np.array([0.27 ,0.54 ,0.82,1.09])

print(F_EH/P_mw_EH)
F_p_W_EH = np.mean(F_EH/P_mw_EH) *1000
print("F_p_W_EH = %.2e kgCO2/W" % F_p_W_EH)

[0.46296296 0.43518519 0.42682927 0.43119266]
F_p_W_EH = 4.39e+02 kgCO2/W


In [34]:
print(F_p_W_EH*1000 / (365*10*24))

5.011900904655508


In [42]:
grid_data = {
"World"                         : 458,  
"ASEAN"                         : 565.01,  
"Africa"                        : 529.88,  
"Asia"                          : 544.4,   
"EU"                            : 209.9,   
"Europe"                        : 282.68,  
"G20"                           : 453.29,  
"G7"                            : 342.42,  
"Latin America and Caribbean"   : 246.97,  
"Middle East"                   : 634.61,  
"North America"                 : 359.95,  
"OECD"                          : 335.93,  
"Oceania"                       : 469.87
}


In [35]:
F_p_kWh_grid = 0.4

F_p_mAh_battery    = (3.65*0.136/3)/3300
F_p_Wh_battery_LP  = 0.03/0.144 # Van Mulders LP
F_p_Wh_battery_MP  = 0.78/3.12 # Van Mulders MP

In [36]:
print("F_p_kWh_grid = %.2e kgCO2/kWh" % F_p_kWh_grid        )
print("F_p_kWh_batt = %.2e kgCO2/kWh" % F_p_mAh_to_p_kWh(F_p_mAh_battery, 1.2))

F_p_kWh_grid = 4.00e-01 kgCO2/kWh
F_p_kWh_batt = 4.18e+01 kgCO2/kWh


In [38]:
print("F_p_J_grid    = %.2e kgCO2/J" % F_p_kWh_to_p_J(F_p_kWh_grid)        )
print("F_p_J_batt    = %.2e kgCO2/J" % F_p_mAh_to_p_J(F_p_mAh_battery, 1.2))
print("F_p_J_batt_LP = %.2e kgCO2/J" % F_p_kWh_to_p_J(F_p_Wh_battery_LP*1000)        )
print("F_p_J_batt_MP = %.2e kgCO2/J" % F_p_kWh_to_p_J(F_p_Wh_battery_MP*1000)        )

F_p_J_grid    = 1.11e-07 kgCO2/J
F_p_J_batt    = 1.16e-05 kgCO2/J
F_p_J_batt_LP = 5.79e-05 kgCO2/J
F_p_J_batt_MP = 6.94e-05 kgCO2/J


In [43]:
F_p_mm2 = 0.026
F_p_mm2 = 0.97/100

A_chip = 0.6

F_p_chip = F_p_mm2 * A_chip
print("F_p_chip = %.2e kgCO2/chip" % F_p_chip)

F_p_chip = 5.82e-03 kgCO2/chip


In [56]:
E_p_pt = 6e-9

F_p_pt_grid = E_p_pt * F_p_kWh_to_p_J(F_p_kWh_grid)        
F_p_pt_batt = E_p_pt * F_p_mAh_to_p_J(F_p_mAh_battery, 1.2)
print("F_p_pt_grid = %.2e kgCO2/pt" % F_p_pt_grid)
print("F_p_pt_batt = %.2e kgCO2/pt" % F_p_pt_batt)

F_p_pt_grid = 6.67e-16 kgCO2/pt
F_p_pt_batt = 6.96e-14 kgCO2/pt


In [57]:
LT_years = 10
LT_s     = LT_years * 365 * 24 * 3600

fs = 10e3

In [58]:
F_op_grid = F_p_pt_grid * fs * LT_s
F_op_batt = F_p_pt_batt * fs * LT_s
F_op_EH   = (F_p_W_EH)  * fs * E_p_pt  


print("F_p_chip = %.2e kgCO2" % F_p_chip)
print("----------------------------")
print("F_op_grid = %.2e kgCO2" % F_op_grid)
print("F_op_batt = %.2e kgCO2" % F_op_batt)
print("F_op_EH   = %.2e kgCO2" % F_op_EH)
print("----------------------------")
print("alpha_grid = %.2e" % (F_p_chip/(F_op_grid+F_p_chip)))
print("alpha_batt = %.2e" % (F_p_chip/(F_op_batt+F_p_chip)))
print("alpha_EH   = %.2e" % (F_p_chip/(F_op_EH  +F_p_chip)))

F_p_chip = 5.82e-03 kgCO2
----------------------------
F_op_grid = 2.10e-03 kgCO2
F_op_batt = 2.20e-01 kgCO2
F_op_EH   = 2.63e-02 kgCO2
----------------------------
alpha_grid = 7.35e-01
alpha_batt = 2.58e-02
alpha_EH   = 1.81e-01
